<a href="https://colab.research.google.com/github/MarcelDiaz/HW2-BigData/blob/main/HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Práctica HW2 Big Data. \
Marcel, Pau, Ane

# **Engineering of Characteristics**

## Duración del trayecto en minutos.

In [11]:
from pyspark.sql.functions import unix_timestamp

# Calculamos la diferencia en segundos, dividimos por 60 y redondeamos a 2 decimales
df_features = df_limpio.withColumn(
    "trip_duration_minutes",
    round((unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60.0, 2)
)

# Mostramos un par de registros para validar que el cálculo tiene sentido
df_features.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes"
).show(5, truncate=False)

+--------------------+---------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+--------------------+---------------------+---------------------+
|2026-01-01 00:54:04 |2026-01-01 00:59:37  |5.55                 |
|2026-01-01 00:15:22 |2026-01-01 00:58:10  |42.8                 |
|2026-01-01 00:47:11 |2026-01-01 01:00:47  |13.6                 |
|2026-01-01 00:17:54 |2026-01-01 00:28:32  |10.63                |
|2026-01-01 00:34:14 |2026-01-01 01:11:58  |37.73                |
+--------------------+---------------------+---------------------+
only showing top 5 rows


## Hora de recogida, día de la semana y mes

In [12]:
df_features = df_features \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_day_of_week", dayofweek(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_month", month(col("tpep_pickup_datetime")))

# Verificamos los resultados
df_features.select(
    "tpep_pickup_datetime",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month"
).show(5)

+--------------------+-----------+------------------+------------+
|tpep_pickup_datetime|pickup_hour|pickup_day_of_week|pickup_month|
+--------------------+-----------+------------------+------------+
| 2026-01-01 00:54:04|          0|                 5|           1|
| 2026-01-01 00:15:22|          0|                 5|           1|
| 2026-01-01 00:47:11|          0|                 5|           1|
| 2026-01-01 00:17:54|          0|                 5|           1|
| 2026-01-01 00:34:14|          0|                 5|           1|
+--------------------+-----------+------------------+------------+
only showing top 5 rows


## Velocidad media del trayecto en millas por hora (mph)

In [13]:
# Calculamos la velocidad en millas por hora (mph)
# Fórmula: (Distancia / Minutos) * 60
# Usamos 'when' para evitar divisiones por cero en caso de que algún viaje haya durado 0 minutos exactos
df_features = df_features.withColumn(
    "avg_speed_mph",
    when(col("trip_duration_minutes") > 0,
         round((col("trip_distance") / col("trip_duration_minutes")) * 60, 2))
    .otherwise(0) # Si el tiempo es 0 o negativo, asignamos 0 a la velocidad
)

# Verificamos los resultados
df_features.select(
    "trip_distance",
    "trip_duration_minutes",
    "avg_speed_mph"
).show(5)

+-------------+---------------------+-------------+
|trip_distance|trip_duration_minutes|avg_speed_mph|
+-------------+---------------------+-------------+
|         0.97|                 5.55|        10.49|
|         5.58|                 42.8|         7.82|
|         2.33|                 13.6|        10.28|
|          1.3|                10.63|         7.34|
|         5.34|                37.73|         8.49|
+-------------+---------------------+-------------+
only showing top 5 rows


## Tarifa por kilómetro

In [14]:
# Factor de conversión: 1 milla = 1.60934 kilómetros
FACTOR_KM = 1.60934

# 1. Creamos la columna de distancia en kilómetros
df_features = df_features.withColumn(
    "trip_distance_km",
    round(col("trip_distance") * FACTOR_KM, 2)
)

# 2. Calculamos la tarifa por kilómetro (Eficiencia)
# Usamos fare_amount para medir estrictamente el coste del viaje (sin propinas ni peajes)
df_features = df_features.withColumn(
    "fare_per_km",
    when(col("trip_distance_km") > 0,
         round(col("fare_amount") / col("trip_distance_km"), 2))
    .otherwise(0)
)

# Verificamos los resultados de las nuevas variables
df_features.select(
    "trip_distance",
    "trip_distance_km",
    "fare_amount",
    "fare_per_km"
).show(5)

+-------------+----------------+-----------+-----------+
|trip_distance|trip_distance_km|fare_amount|fare_per_km|
+-------------+----------------+-----------+-----------+
|         0.97|            1.56|        7.2|       4.62|
|         5.58|            8.98|       38.7|       4.31|
|         2.33|            3.75|       14.2|       3.79|
|          1.3|            2.09|       11.4|       5.45|
|         5.34|            8.59|       37.3|       4.34|
+-------------+----------------+-----------+-----------+
only showing top 5 rows


# *Exploratory Data Analysis with Spark*

In [15]:
# ¿Cuáles son las horas con más recogidas?
horas_mayor_actividad = df_features.groupBy("pickup_hour") \
    .count() \
    .orderBy(col("count").desc())


# Mostrar los resultados (las 24 horas del día ordenadas por demanda)
print("Horas con mayor cantidad de recogidas:")
horas_mayor_actividad.show(24)

Horas con mayor cantidad de recogidas:
+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|554301|
|         17|547918|
|         16|511107|
|         15|501526|
|         19|481508|
|         14|474359|
|         20|441442|
|         13|440046|
|         21|439965|
|         12|425039|
|         11|392272|
|         22|371059|
|         10|364963|
|          9|336167|
|          8|287494|
|         23|266776|
|          7|205753|
|          0|184893|
|          1|119736|
|          6|103769|
|          2| 76632|
|          3| 53848|
|          5| 49241|
|          4| 38821|
+-----------+------+



In [16]:
# ¿Cuáles son las zonas de recogida y de destino más activas?
# Agrupamos por ID de recogida y contamos
zonas_recogida_activas = df_features.groupBy("PULocationID").count().orderBy(col("count").desc())

print("10 zonas de recogida más activas:")
zonas_recogida_activas.show(10)

# Agrupamos por ID de destino y contamos
zonas_destino_activas = df_features.groupBy("DOLocationID").count().orderBy(col("count").desc())

print("10 zonas de destino más activas:")
zonas_destino_activas.show(10) 

10 zonas de recogida más activas:
+------------+------+
|PULocationID| count|
+------------+------+
|         237|408832|
|         132|403958|
|         236|364018|
|         161|361542|
|         186|284105|
|         162|277420|
|         142|253088|
|         138|245394|
|         230|245116|
|         163|217247|
+------------+------+
only showing top 10 rows
10 zonas de destino más activas:
+------------+------+
|DOLocationID| count|
+------------+------+
|         236|385408|
|         237|362424|
|         161|277638|
|         239|226898|
|         142|222148|
|         141|221776|
|         170|218596|
|         230|212483|
|         162|206414|
|          68|188931|
+------------+------+
only showing top 10 rows


In [17]:
# ¿Cómo varía la demanda por día de la semana?

demanda_semanal =  df_features.groupBy("pickup_day_of_week").count().orderBy("pickup_day_of_week")

print("Demanda por día de la semana:")
demanda_semanal.show()

Demanda por día de la semana:
+------------------+-------+
|pickup_day_of_week|  count|
+------------------+-------+
|                 1| 873956|
|                 2| 906188|
|                 3|1144791|
|                 4|1124299|
|                 5|1260287|
|                 6|1212294|
|                 7|1146820|
+------------------+-------+



In [18]:
# ¿Qué zonas generan más ingresos totales?
ingresos_zonas = df_features.groupBy("PULocationID").sum("total_amount").orderBy(col("sum(total_amount)").desc())
print("10 zonas con más ingresos:")
ingresos_zonas.show(10)

10 zonas con más ingresos:
+------------+-------------------+
|PULocationID|  sum(total_amount)|
+------------+-------------------+
|         132|3.226628015000224E7|
|         138|1.715570391999912E7|
|         161|  9277362.280000083|
|         237|   8515174.26000005|
|         236|  7676431.900000242|
|         186|  7259116.609999913|
|         230|  6909554.159999854|
|         162|  6902127.079999959|
|         142|  5634253.689999989|
|         163|  5409468.019999962|
+------------+-------------------+
only showing top 10 rows


In [19]:
# ¿Cuál es la distancia media de trayecto por hora?
distancia_media_por_hora = df_features.groupBy("pickup_hour").avg("trip_distance").withColumnRenamed("avg(trip_distance)", "distancia_media").orderBy("pickup_hour")

print("Distancia media de trayecto por hora:")
distancia_media_por_hora.show(24)

Distancia media de trayecto por hora:
+-----------+------------------+
|pickup_hour|   distancia_media|
+-----------+------------------+
|          0|4.0103672935157135|
|          1| 3.490133460279275|
|          2| 3.141387801440654|
|          3| 4.019329222998064|
|          4| 5.046239148914259|
|          5| 6.636811397006561|
|          6| 5.484369898524613|
|          7|3.9853173465271308|
|          8|3.3634024710081136|
|          9| 3.484605359836019|
|         10| 3.267125900433774|
|         11| 3.089897469103079|
|         12|3.1877762275932167|
|         13|3.2993872458788593|
|         14|3.5143588294941277|
|         15|3.4946948114355334|
|         16| 3.493790830491442|
|         17|3.1158681043513754|
|         18|2.9498817249112226|
|         19| 3.229923178846466|
|         20| 3.501894495766144|
|         21|3.4996867705385633|
|         22|3.7202640280925605|
|         23| 4.085124973760757|
+-----------+------------------+



In [20]:
# ¿Cuál es la tarifa media por día de la semana?

tarifa_media_dia = df_features.groupBy("pickup_day_of_week") \
    .agg(F.avg(col("fare_amount")).alias("tarifa_media_base")).orderBy("pickup_day_of_week")

print("Tarifa media por día de la semana:")
tarifa_media_dia.show()

Tarifa media por día de la semana:
+------------------+------------------+
|pickup_day_of_week| tarifa_media_base|
+------------------+------------------+
|                 1|19.956179418643107|
|                 2|20.443384187386872|
|                 3|19.849513168778916|
|                 4|19.711905098198734|
|                 5|19.839452093054618|
|                 6|19.506694448705826|
|                 7| 18.07172703650052|
+------------------+------------------+



In [21]:
# ¿Qué tipo de pago es el más frecuente?
# 1. Agrupar por tipo de pago y contar
pago_frecuente = df_features.groupBy("payment_type").count()

# 2. Usar withColumn para traducir los códigos a etiquetas legibles
# (Basado en el diccionario estándar de la TLC de NYC)
pago_frecuente = pago_frecuente.withColumn("metodo_pago", 
    when(col("payment_type") == 1, "Tarjeta de Crédito")
    .when(col("payment_type") == 2, "Efectivo")
    .when(col("payment_type") == 3, "Sin cargo")
    .when(col("payment_type") == 4, "Disputa")
    .otherwise("Otros")
)

# 3. Ordenar por la cantidad de mayor a menor
pago_frecuente = pago_frecuente.select("metodo_pago", "count") \
    .orderBy(col("count").desc())

print("Frecuencia por tipo de pago:")
pago_frecuente.show()

Frecuencia por tipo de pago:
+------------------+-------+
|       metodo_pago|  count|
+------------------+-------+
|Tarjeta de Crédito|6706617|
|          Efectivo| 877162|
|           Disputa|  62113|
|         Sin cargo|  22743|
+------------------+-------+



In [22]:
# ¿Qué zonas presentan la mayor duración media de viaje?
# 1. Agrupar por zona de recogida y calcular la media de duración
duracion_por_zona = df_features.groupBy("PULocationID").avg("trip_duration_minutes")

# 2. Usar withColumn para redondear y limpiar el nombre de la columna
duracion_por_zona = duracion_por_zona \
    .withColumn("duracion_media_min", round(col("avg(trip_duration_minutes)"), 2)) \
    .select("PULocationID", "duracion_media_min") \
    .orderBy(col("duracion_media_min").desc())

print("10 zonas con viajes de mayor duración media:")
duracion_por_zona.show(10)


10 zonas con viajes de mayor duración media:
+------------+------------------+
|PULocationID|duracion_media_min|
+------------+------------------+
|           5|            157.75|
|          86|              83.9|
|         117|             81.67|
|         201|             75.92|
|          19|              70.1|
|         207|             69.58|
|         203|             69.21|
|         205|             67.52|
|         139|              67.3|
|          38|             66.77|
+------------+------------------+
only showing top 10 rows


In [23]:
# ¿Qué zonas tienen trayectos cortos pero muy frecuentes?
# 1. Definimos el umbral de lo que consideramos "trayecto corto" (ej. 2 km)
umbral_km = 2.0

# 2. Filtramos los datos para quedarnos solo con esos viajes cortos
df_cortos = df_features.filter(col("trip_distance_km") < umbral_km)

# 3. Agrupamos por zona de recogida (PULocationID) y contamos la frecuencia
zonas_cortas_frecuentes = df_cortos.groupBy("PULocationID").count()

# 4. Renombramos la columna para que sea clara y ordenamos de mayor a menor
zonas_cortas_frecuentes = zonas_cortas_frecuentes \
    .withColumn("numero_viajes_cortos", col("count")) \
    .select("PULocationID", "numero_viajes_cortos") \
    .orderBy(col("numero_viajes_cortos").desc())

print(f"Top 10 Zonas con trayectos cortos (< {umbral_km} km) más frecuentes:")
zonas_cortas_frecuentes.show(10)

Top 10 Zonas con trayectos cortos (< 2.0 km) más frecuentes:
+------------+--------------------+
|PULocationID|numero_viajes_cortos|
+------------+--------------------+
|         237|              214639|
|         236|              165259|
|         161|              145143|
|         162|              111194|
|         186|              111063|
|         230|               99467|
|         141|               95668|
|         142|               93125|
|         234|               90559|
|         163|               88460|
+------------+--------------------+
only showing top 10 rows


In [24]:
# Identificad trayectos con velocidad media irreal.
# 1. Definimos un umbral de velocidad "irreal" (ej. 80 mph)
umbral_velocidad = 80

# 2. Filtramos los registros que superan ese umbral
trayectos_irreales = df_features.filter(col("avg_speed_mph") > umbral_velocidad)

# 3. Seleccionamos las columnas clave para inspeccionar por qué son irreales
# Usamos withColumn para ver también la velocidad en km/h si lo prefieres
trayectos_irreales = trayectos_irreales \
    .withColumn("velocidad_kmh", round(col("avg_speed_mph") * 1.60934, 2)) \
    .select(
        "PULocationID", 
        "DOLocationID", 
        "trip_distance", 
        "trip_duration_minutes", 
        "avg_speed_mph",
        "velocidad_kmh"
    ) \
    .orderBy(col("avg_speed_mph").desc())

print(f"Trayectos con velocidad media superior a {umbral_velocidad} mph:")
trayectos_irreales.show(10)

# Para el informe: Contar cuántos registros hay con este problema
total_irreales = trayectos_irreales.count()
print(f"Total de registros con velocidad irreal: {total_irreales}")

Trayectos con velocidad media superior a 80 mph:
+------------+------------+-------------+---------------------+-------------+-------------+
|PULocationID|DOLocationID|trip_distance|trip_duration_minutes|avg_speed_mph|velocidad_kmh|
+------------+------------+-------------+---------------------+-------------+-------------+
|         179|         146|     98773.58|                  6.1|    971543.41|   1563543.67|
|         138|         170|     36610.27|                15.35|    143102.03|    230299.82|
|         231|         159|     32775.53|                26.02|      75577.7|    121630.22|
|         132|          79|        18.74|                 0.02|      56220.0|     90477.09|
|         236|         141|      7604.17|                 8.28|     55102.68|     88678.95|
|         164|          68|      8301.22|                 9.62|     51774.76|     83323.19|
|         145|         145|         45.3|                 0.07|     38828.57|     62488.37|
|         145|         145|    

In [25]:
# Identificad registros con tarifa sospechosamente alta para una distancia corta.
umbral_tarifa = 50.0  # Tarifas mayores a 50$
umbral_distancia_km = 1.0 # Distancias menores a 1 km

# 2. Filtramos los registros que cumplen ambas condiciones
anomalias_tarifa = df_features.filter(
    (col("fare_amount") > umbral_tarifa) & 
    (col("trip_distance_km") < umbral_distancia_km)
)

# 3. Seleccionamos las columnas para auditar el caso
# Ordenamos por tarifa descendente para ver los casos más "extremos" primero
resultado_anomalias = anomalias_tarifa.select(
    "PULocationID", 
    "DOLocationID", 
    "trip_distance_km", 
    "fare_amount", 
    "trip_duration_minutes"
).orderBy(col("fare_amount").desc())

print(f"Registros con tarifa > {umbral_tarifa}$ y distancia < {umbral_distancia_km} km:")
resultado_anomalias.show(10)

Registros con tarifa > 50.0$ y distancia < 1.0 km:
+------------+------------+----------------+-----------+---------------------+
|PULocationID|DOLocationID|trip_distance_km|fare_amount|trip_duration_minutes|
+------------+------------+----------------+-----------+---------------------+
|         178|          22|            0.64|      950.0|                 1.17|
|          76|          63|            0.05|      800.0|                 0.13|
|          16|          64|            0.21|      740.0|                  0.2|
|         226|         226|            0.24|      650.0|                 1.12|
|         121|          98|             0.1|      540.0|                  0.1|
|          14|          14|            0.08|      500.0|                 0.08|
|          34|          97|            0.58|      500.0|                 2.47|
|          19|          19|            0.03|      498.0|                 0.15|
|          10|          10|            0.02|      490.0|                  0.1|
|

In [26]:
# Clasificad las 10 zonas de recogida principales por mes.
# 1. Agrupar por mes y zona para contar cuántos viajes hay en cada combinación
conteo_mensual = df_features.groupBy("pickup_month", "PULocationID").count()

# 2. Definir la especificación de la ventana para el ranking por mes
window_spec = Window.partitionBy("pickup_month").orderBy(desc("count"))
top_10_por_mes = conteo_mensual.withColumn("ranking", rank().over(window_spec))

# 4. Filtramos: solo ranking del 1 al 10 Y solo meses 1, 2 y 3 (Ene, Feb, Mar)
resultado_trimestre = top_10_por_mes.filter(
    (col("ranking") <= 10) & 
    (col("pickup_month").isin(1, 2, 3))
).orderBy("pickup_month", "ranking")

# 5. Imprimimos los resultados
meses = {1: "Enero", 2: "Febrero", 3: "Marzo"}

for mes_id, nombre in meses.items():
    print(f"Top 10 zonas de {nombre}:")
    top_10_por_mes.filter((col("pickup_month") == mes_id) & (col("ranking") <= 10)) \
        .select("ranking", "PULocationID", "count") \
        .orderBy("ranking") \
        .show()

Top 10 zonas de Enero:
+-------+------------+------+
|ranking|PULocationID| count|
+-------+------------+------+
|      1|         132|141089|
|      2|         237|134814|
|      3|         236|122216|
|      4|         161|117617|
|      5|         186| 94078|
|      6|         162| 90668|
|      7|         142| 86748|
|      8|         230| 82995|
|      9|         138| 80590|
|     10|         239| 70633|
+-------+------------+------+

Top 10 zonas de Febrero:
+-------+------------+------+
|ranking|PULocationID| count|
+-------+------------+------+
|      1|         237|121670|
|      2|         132|116542|
|      3|         236|110375|
|      4|         161|106430|
|      5|         186| 83616|
|      6|         162| 81859|
|      7|         142| 73752|
|      8|         138| 72819|
|      9|         230| 68890|
|     10|         163| 63834|
+-------+------------+------+

Top 10 zonas de Marzo:
+-------+------------+------+
|ranking|PULocationID| count|
+-------+------------+-----

In [27]:
# Comparad el comportamiento de los trayectos de día frente a noche.
# 1. Crear la columna de segmento (Día vs Noche)
df_dia_noche = df_features.withColumn("segmento_horario", 
    when((col("pickup_hour") >= 6) & (col("pickup_hour") < 20), "Día")
    .otherwise("Noche")
)

# 2. Agrupar por el segmento y calcular métricas comparativas
comparativa = df_dia_noche.groupBy("segmento_horario").agg(
    F.count("*").alias("total_viajes"),
    round(F.avg("trip_distance_km"), 2).alias("distancia_media_km"),
    round(F.avg("fare_amount"), 2).alias("tarifa_media"),
    round(F.avg("avg_speed_mph"), 2).alias("velocidad_media_mph"),
    round(F.avg("tip_amount"), 2).alias("propina_media")
)

print("Comparativa de comportamiento: Día vs Noche")
comparativa.show()

Comparativa de comportamiento: Día vs Noche
+----------------+------------+------------------+------------+-------------------+-------------+
|segmento_horario|total_viajes|distancia_media_km|tarifa_media|velocidad_media_mph|propina_media|
+----------------+------------+------------------+------------+-------------------+-------------+
|           Noche|     2042413|              6.06|       19.67|              13.44|         3.79|
|             Día|     5626222|              5.39|       19.56|              10.19|         3.55|
+----------------+------------+------------------+------------+-------------------+-------------+



In [28]:
# Encontrad patrones de concentración de demanda a lo largo del tiempo.
concentracion_demanda = df_features.groupBy("pickup_day_of_week", "pickup_hour").count()

# 2. Ordenar para ver los momentos de máxima saturación del sistema
# Usamos withColumn para renombrar el conteo y hacerlo más claro
concentracion_demanda = concentracion_demanda \
    .withColumn("total_viajes", col("count")) \
    .select("pickup_day_of_week", "pickup_hour", "total_viajes") \
    .orderBy(col("total_viajes").desc())

print("Top 15 de momentos con mayor concentración de demanda:")
concentracion_demanda.show(15)

Top 15 de momentos con mayor concentración de demanda:
+------------------+-----------+------------+
|pickup_day_of_week|pickup_hour|total_viajes|
+------------------+-----------+------------+
|                 5|         18|       94126|
|                 5|         17|       91090|
|                 6|         18|       89073|
|                 3|         18|       88541|
|                 3|         17|       87082|
|                 6|         17|       85966|
|                 4|         18|       85611|
|                 5|         19|       82554|
|                 4|         17|       82469|
|                 5|         16|       82095|
|                 5|         15|       81063|
|                 5|         21|       80360|
|                 6|         16|       80004|
|                 6|         15|       79731|
|                 5|         20|       78488|
+------------------+-----------+------------+
only showing top 15 rows


# Uso de Funciones de Ventana

## Ranking de las principales zonas de recogida por día o por mes

In [29]:
# Agrupamos los datos para contar los viajes por cada mes y zona de recogida
df_viajes_zona_mes = df_features.groupBy("pickup_month", "PULocationID") \
    .agg(count("*").alias("total_viajes"))

# Definimos la ventana: Partición por 'pickup_month' y ordenamos por 'total_viajes' de mayor a menor
window_ranking_mes = Window.partitionBy("pickup_month").orderBy(desc("total_viajes"))

# Aplicamos la función rank() sobre nuestra ventana para crear una clasificación
df_ranking_mes = df_viajes_zona_mes.withColumn(
    "ranking_zona", 
    rank().over(window_ranking_mes)
)

# Filtramos para obtener solo el Top 5 de zonas por cada mes y ordenamos para visualizar
print("Top 5 de las principales zonas de recogida por mes:")
df_ranking_mes.filter(col("ranking_zona") <= 5) \
    .orderBy("pickup_month", "ranking_zona") \
    .show()


Top 5 de las principales zonas de recogida por mes:
+------------+------------+------------+------------+
|pickup_month|PULocationID|total_viajes|ranking_zona|
+------------+------------+------------+------------+
|           1|         132|      141089|           1|
|           1|         237|      134814|           2|
|           1|         236|      122216|           3|
|           1|         161|      117617|           4|
|           1|         186|       94078|           5|
|           2|         237|      121670|           1|
|           2|         132|      116542|           2|
|           2|         236|      110375|           3|
|           2|         161|      106430|           4|
|           2|         186|       83616|           5|
|           3|         237|      152348|           1|
|           3|         132|      146327|           2|
|           3|         161|      137495|           3|
|           3|         236|      131427|           4|
|           3|         186|   

## Ranking de zonas con mayor ingreso por período.

In [30]:
# Agregupamos los datos: Sumamos el ingreso total por cada mes y zona de recogida
df_ingresos_zona = df_features.groupBy("pickup_month", "PULocationID").agg(
    round(sum("total_amount"), 2).alias("ingreso_total")
)

# Definimos la ventana, particionamos por mes y ordenamos por el ingreso total de forma descendente 
ventana_ranking = Window.partitionBy("pickup_month").orderBy(col("ingreso_total").desc())

# Aplicamos la función de ventana dense_rank() para generar una clasificación
df_ranking = df_ingresos_zona.withColumn(
    "ranking_ingresos", 
    dense_rank().over(ventana_ranking)
)

# Filtramos para ver solo el Top 5 de cada mes y mostramos el resultado
print("Top 5 de zonas con mayor ingreso por mes:")
df_ranking.filter(col("ranking_ingresos") <= 5) \
          .orderBy("pickup_month", "ranking_ingresos") \
          .show(15)

Top 5 de zonas con mayor ingreso por mes:
+------------+------------+-------------+----------------+
|pickup_month|PULocationID|ingreso_total|ranking_ingresos|
+------------+------------+-------------+----------------+
|           1|         132|1.118076479E7|               1|
|           1|         138|   5589586.13|               2|
|           1|         161|   2959015.46|               3|
|           1|         237|    2762806.6|               4|
|           1|         236|   2558471.93|               5|
|           2|         132|    9300676.3|               1|
|           2|         138|   5120883.79|               2|
|           2|         161|    2763680.1|               3|
|           2|         237|   2572344.78|               4|
|           2|         236|   2348630.08|               5|
|           3|         132|1.178483906E7|               1|
|           3|         138|    6445234.0|               2|
|           3|         161|   3554666.72|               3|
|           3|

# **Análisis de Rendimiento**

In [31]:
import time

# 1A OPTIMIZACIÓN: Uso de CACHÉ sobre la operación groupBy("PULocationID", "pickup_hour").count()
# En el apartado de Análisis Exploratorio con Spark
print("--- INICIANDO OPTIMIZACIÓN 1: CACHÉ ---")

# Consulta SIN caché (fuerza a Spark a calcular todo desde el origen)
df_features.unpersist() # Nos aseguramos de que no esté en caché

inicio = time.time()
resultado_sin_cache = df_features.groupBy("PULocationID", "pickup_hour").count().count()
fin = time.time()
tiempo_sin = fin - inicio

# Usamos :.2f para que Python imprima solo 2 decimales sin usar la función round()
print(f"Tiempo de ejecución SIN cache: {tiempo_sin:.2f} segundos")

# Aplicamos CACHÉ
df_features.cache()

# Ejecutamos una acción para que se guarde físicamente en la memoria RAM
_ = df_features.count() 

# Consulta CON caché (los datos ya están listos en la memoria)
inicio2 = time.time()
resultado_con_cache = df_features.groupBy("PULocationID", "pickup_hour").count().count()
fin2 = time.time()
tiempo_con = fin2 - inicio2
print(f"Tiempo de ejecución CON cache: {tiempo_con:.2f} segundos")

# Cálculo de la mejora
mejora_pct = ((tiempo_sin - tiempo_con) / tiempo_sin) * 100
print(f"Porcentaje de mejora en el rendimiento: {mejora_pct:.2f}%\n")


# 2a OPTIMIZACIÓN: Reducción de particiones de Shuffle 
# con la operación groupBy("PULocationID", "pickup_month").agg(sum("total_amount"))
# en el apartado Análisis Exploratorio con Spark
print("--- INICIANDO OPTIMIZACIÓN 2: SHUFFLE PARTITIONS ---")

# Aseguramos la configuración por defecto de Spark (200)
spark.conf.set("spark.sql.shuffle.partitions", "200")

inicio_a = time.time()
res_a = df_features.groupBy("PULocationID", "pickup_month").agg(sum("total_amount")).count()
fin_a = time.time()
tiempo_a = fin_a - inicio_a
print(f"Tiempo con 200 particiones (default): {tiempo_a:.2f} segundos")

# Optimizamos reduciendo las particiones a un número más eficiente (ej: 16)
spark.conf.set("spark.sql.shuffle.partitions", "16")

inicio_b = time.time()
res_b = df_features.groupBy("PULocationID", "pickup_month").agg(sum("total_amount")).count()
fin_b = time.time()
tiempo_b = fin_b - inicio_b
print(f"Tiempo con 16 particiones: {tiempo_b:.2f} segundos")

mejora_c = ((tiempo_a - tiempo_b) / tiempo_a) * 100
print(f"Mejora por Shuffle: {mejora_c:.2f}%\n")


# 3a OPTIMIZACIÓN: Repartición de datos con la operación Window.partitionBy("pickup_month") con rank()
# En el apartado de Funciones de Ventana
print("--- INICIANDO OPTIMIZACIÓN 3: REPARTITION VS WINDOW ---")

# Preparamos un dataframe agrupado pequeño para hacer el ranking
df_agrupado = df_features.groupBy("pickup_month", "PULocationID").agg(sum("total_amount").alias("ingresos"))
window_spec = Window.partitionBy("pickup_month").orderBy(desc("ingresos"))

# Ejecución normal SIN reparticionar
inicio_d = time.time()
res_d = df_agrupado.withColumn("rank", rank().over(window_spec)).count()
fin_d = time.time()
tiempo_d = fin_d - inicio_d
print(f"Tiempo SIN repartition previo: {tiempo_d:.2f} segundos")

# Optimizamos: Reparticionamos físicamente por el mes antes de hacer la ventana
df_reparticionado = df_agrupado.repartition("pickup_month")

inicio_e = time.time()
res_e = df_reparticionado.withColumn("rank", rank().over(window_spec)).count()
fin_e = time.time()
tiempo_e = fin_e - inicio_e
print(f"Tiempo CON repartition previo: {tiempo_e:.2f} segundos")

mejora_f = ((tiempo_d - tiempo_e) / tiempo_d) * 100
print(f"Mejora por Repartition: {mejora_f:.2f}%")

--- INICIANDO OPTIMIZACIÓN 1: CACHÉ ---
26/05/10 10:59:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
Tiempo de ejecución SIN cache: 4.64 segundos
Tiempo de ejecución CON cache: 1.02 segundos
Porcentaje de mejora en el rendimiento: 77.98%

--- INICIANDO OPTIMIZACIÓN 2: SHUFFLE PARTITIONS ---
Tiempo con 200 particiones (default): 0.46 segundos
Tiempo con 16 particiones: 0.37 segundos
Mejora por Shuffle: 20.26%

--- INICIANDO OPTIMIZACIÓN 3: REPARTITION VS WINDOW ---
Tiempo SIN repartition previo: 0.79 segundos
Tiempo CON repartition previo: 0.44 segundos
Mejora por Repartition: 44.61%


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=8d0dc42f-e8ad-49d6-a082-9cd0361c7d97' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>